# Session 9: Naive Bayes Classification and Support Vector Machines (I)

## Naive Bayes Classification

In [ ]:
"""
Demo for Naive Bayes classification.

This demo uses numeric vectors and not text/tokens so that we can directly
visualize the classification boundary.
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB

# 1. Define a neutral dataset with two numerical features (X0, X1)
# Class 0: Centered around (2, 2) | Class 1: Centered around (6, 6)
X_train = np.array([
    [1.5, 2.0], [2.0, 1.5], [2.5, 2.5], [1.8, 2.2],  # Class 0 points
    [5.5, 6.0], [6.0, 5.5], [6.5, 6.5], [5.8, 6.2]   # Class 1 points
])
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# 2. Train the Gaussian Naive Bayes Model
model = GaussianNB()
model.fit(X_train, y_train)

# 3. Extract and display the Prior Probabilities P(Class)
priors = model.class_prior_
print("--- Class Prior Estimates ---")
print(f"P(Class 0): {priors[0]:.2f}")
print(f"P(Class 1): {priors[1]:.2f}\n")

# 4. Predict on a new query point and evaluate Posterior Probabilities P(Class | X)
X_query = np.array([[4.0, 4.2]])
posteriors = model.predict_proba(X_query)[0]
prediction = model.predict(X_query)[0]

print("--- Query Point Evaluation ---")
print(f"Query Coordinate: {X_query[0]}")
print(f"Posterior P(Class 0 | Query): {posteriors[0]:.4f}")
print(f"Posterior P(Class 1 | Query): {posteriors[1]:.4f}")
print(f"Assigned Classification: Class {prediction}\n")

# 5. Generate a dense grid to plot the decision boundary background
x_min, x_max = 0, 8
y_min, y_max = 0, 8
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                     np.arange(y_min, y_max, 0.05))

# Predict classifications across the entire coordinate grid
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# 6. Create the visualization plot
plt.figure(figsize=(7, 6), facecolor='white')

# 1. Correctly fill both background regions by explicitly defining data levels
plt.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], alpha=0.15, colors=['blue', 'orange'])

# 2. Draw the exact decision threshold line right at the 0.5 midpoint
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5, linestyles='dashed')
# Draw the exact decision threshold boundary line
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5, linestyles='dashed')
# Scatter plot the training points
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], color='blue', label='Class 0', edgecolors='k')
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], color='orange', label='Class 1', edgecolors='k')

# Highlight the custom query point
plt.scatter(X_query[0, 0], X_query[0, 1], color='red', marker='X', s=150, label=f'Query Point (Class {prediction})', edgecolors='k')

# Configure labels and limits
plt.title("Gaussian Naive Bayes Decision Boundary & Posteriors")
plt.xlabel("Feature X0")
plt.ylabel("Feature X1")
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right')

plt.tight_layout()
plt.savefig("naive_bayes.png", dpi=150)
print("Saved plot to svm_qp_geometry.png")


## Support Vector Machines
### (I) Hard-margin: Data is linearily separable

In [ ]:
"""
Demo of the linear (hard-margin) Support Vector Machine, implemented
explicitly by solving the quadratic program.

Model:      y_hat = sign(w^T x + b)
Objective:  min_{w,b}  (1/2) ||w||^2
Constraints: y_i (w^T x_i + b) >= 1   for all i

This is solved directly as a constrained quadratic program via
scipy.optimize.minimize (SLSQP) - no closed-form shortcut - so the
same code generalises to any small, separable dataset.

Three synthetic datasets are generated with different class
separation (the "gap" between the two clusters at generation time),
to show how the geometry of the fitted SVM responds to how well
separable the data is. The margin achieved by the fitted
model is:

    margin = 2 / ||w||

A second figure demonstrates how the optimum is found: for one of the
datasets, the quadratic objective (1/2)||w||^2 is drawn as a contour
plot directly in (w1, w2)-space (with b fixed at its optimal value),
together with the linear constraints from the active support vectors.
The optimal w sits on the smallest objective contour that
still touches the feasible region - geometrically, where the circular
contour of the objective is tangent to the constraint boundary.

A third section extends this to the SOFT-margin case: a NOISE_FRAC
fraction of points have their label flipped after generation, which
makes the resulting data linearly NON-separable (no hyperplane can
satisfy every hard-margin constraint). The soft-margin primal QP,

    min_{w,b,xi}  (1/2)||w||^2 + C * sum(xi)
    s.t.  y_i (w^T x_i + b) >= 1 - xi_i,   xi_i >= 0

is solved the same way, via SLSQP, for a grid of C values. Data is
split into train/holdout/test: C is selected by holdout accuracy,
then the chosen model is refit on train+holdout and evaluated once,
honestly, on the untouched test set.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

N_PER_CLASS = 50
GAPS = [.2, 2.5, 5.0]     # class separation used to generate each dataset
SPREAD = 0.4                # within-class scatter (kept small relative to GAPS
                             # so every dataset generated below is separable)
SEED = 0

# --- Soft-margin extension: non-separable data, C selection, hold-out/test ---
N_PER_CLASS_SOFT = 40       # bigger dataset, so a train/holdout/test split is meaningful
GAP_SOFT = 2.0
NOISE_FRAC = 0.2            # fraction of points with a flipped (wrong) label -
                             # this is what makes the data linearly NON-separable
C_GRID = [0.01, 0.1, 1, 10, 100]  # candidate values of the slack penalty C


def generate_separable_data(n_per_class, gap, spread=SPREAD, seed=SEED, max_attempts=200):
    """Two small 2D clusters, separated along the x1-axis by `gap`
    (centre-to-centre distance), each with Gaussian scatter of
    standard deviation `spread`.

    Separability is GUARANTEED, not just likely: `spread` is chosen to
    be well below `gap` by default, but for a large `n_per_class` even
    a small spread has some chance of producing an overlapping tail
    point. Instead of leting that surface rash in fit_svm_primal, 
    this function checks separability itself (the same LinearSVC check 
    used there) and, if the draw isn't separable, retries with a new seed. 
    it still isn't separable after `max_attempts` tries, `spread` is shrunk 
    and the seed loop restarts - this always terminates, since a small enough 
    spread is separable for any finite n_per_class and gap > 0."""
    attempt = 0
    current_spread = spread
    while True:
        rng = np.random.default_rng(seed + attempt)
        mu_neg = np.array([-gap / 2, 0.0])
        mu_pos = np.array([gap / 2, 0.0])
        X_neg = rng.normal(mu_neg, current_spread, size=(n_per_class, 2))
        X_pos = rng.normal(mu_pos, current_spread, size=(n_per_class, 2))
        X = np.vstack([X_neg, X_pos])
        y = np.concatenate([-np.ones(n_per_class), np.ones(n_per_class)])

        if check_separable(X, y):
            if current_spread != spread:
                print(f"  (generate_separable_data: reduced spread from {spread} to "
                      f"{current_spread:.4f} to guarantee separability at "
                      f"n_per_class={n_per_class}, gap={gap})")
            return X, y

        attempt += 1
        if attempt >= max_attempts:
            # New seeds alone aren't finding a separable draw at this
            # spread - shrink it and start the seed search over. This
            # always converges: a small enough spread is separable.
            current_spread *= 0.8
            attempt = 0


def check_separable(X, y):
    """Quick feasibility check for the hard-margin problem: fit an
    unregularised linear model (LinearSVC with a very large C, which
    approximates hard-margin) and see whether it achieves perfect
    accuracy. If not, no hyperplane separates the classes and the
    hard-margin QP has no feasible (w, b) at all - no solver can
    converge to a solution that doesn't exist, so it's worth checking
    this directly rather than reading solver failure as a bug."""
    from sklearn.svm import LinearSVC
    with warnings_ignored():
        clf = LinearSVC(C=1e6, max_iter=100000, tol=1e-6)
        clf.fit(X, y)
    return np.mean(clf.predict(X) == y) == 1.0


class warnings_ignored:
    def __enter__(self):
        import warnings
        self._cm = warnings.catch_warnings()
        self._cm.__enter__()
        warnings.simplefilter("ignore")
        return self

    def __exit__(self, *args):
        self._cm.__exit__(*args)


def fit_svm_primal(X, y, x0=None):
    """Solve the hard-margin SVM primal QP directly:
        min_{w,b} 0.5*(w1^2+w2^2)   s.t.  y_i*(w.x_i+b) - 1 >= 0
    via SLSQP. Returns (w, b).

    x0, if given, warm-starts the solver (see fit_svm_soft for why).
    Multiple starting points are tried as a fallback, same as the
    soft-margin solver, since SLSQP is a general local optimiser, not
    a dedicated QP solver, and becomes less numerically reliable as
    the dataset grows or points sit close to the boundary.

    IMPORTANT: hard-margin SVM has a feasible solution ONLY if the
    data is genuinely linearly separable. If it isn't, this function
    will fail regardless of solver robustness - that failure means
    "no separating hyperplane exists", not "the optimiser is broken".
    Use fit_svm_soft for data that may not be perfectly separable."""
    n, d = X.shape

    def objective(params):
        w = params[:d]
        return 0.5 * np.dot(w, w)

    def objective_grad(params):
        w = params[:d]
        return np.concatenate([w, [0.0]])

    def constraint_fn(params):
        w, b = params[:d], params[d]
        return y * (X @ w + b) - 1.0

    def constraint_jac(params):
        w, b = params[:d], params[d]
        # d/dw_k [y_i*(w.x_i+b) - 1] = y_i * x_i_k ; d/db [...] = y_i
        J = np.column_stack([y[:, None] * X, y])
        return J

    starts = [x0] if x0 is not None else []
    starts.append(np.zeros(d + 1))
    last_error = None
    for start in starts:
        result = minimize(
            objective, start, jac=objective_grad, method="SLSQP",
            constraints=[{"type": "ineq", "fun": constraint_fn, "jac": constraint_jac}],
            options={"maxiter": 1000, "ftol": 1e-12},
        )
        if result.success:
            return result.x[:d], result.x[d]
        last_error = result.message

    # SLSQP failed from every start. Try trust-constr before concluding
    # anything about feasibility - see fit_svm_soft for why.
    from scipy.optimize import NonlinearConstraint, Bounds as ScipyBounds
    scipy_bounds = ScipyBounds([-np.inf] * (d + 1), [np.inf] * (d + 1))
    nlc = NonlinearConstraint(constraint_fn, 0, np.inf, jac=constraint_jac)
    result = minimize(
        objective, starts[0], jac=objective_grad, method="trust-constr",
        bounds=scipy_bounds, constraints=[nlc],
        options={"maxiter": 3000, "gtol": 1e-8, "xtol": 1e-10},
    )
    if result.success:
        return result.x[:d], result.x[d]

    # Every solver/start failed. Distinguish "the data genuinely isn't
    # separable" (expected, fixable only by switching to soft-margin)
    # from "the solvers just struggled numerically" (a feasible
    # solution does exist here, but wasn't found).
    if not check_separable(X, y):
        raise RuntimeError(
            "Hard-margin SVM has no solution: this data is not linearly "
            "separable (confirmed via a quick LinearSVC check). Use "
            "fit_svm_soft instead, or regenerate with a larger GAP / "
            "smaller SPREAD / fewer points."
        )
    raise RuntimeError(
        f"SVM QP did not converge despite the data being separable "
        f"(SLSQP: {last_error}; trust-constr: {result.message}). Try "
        f"reducing N_PER_CLASS / SPREAD for a better-conditioned problem."
    )


def find_support_vectors(X, y, w, b, tol=1e-3):
    margin_values = y * (X @ w + b)
    is_sv = np.abs(margin_values - 1.0) < tol
    return is_sv


def plot_svm(ax, X, y, w, b, gap):
    is_sv = find_support_vectors(X, y, w, b)
    margin = 2.0 / np.linalg.norm(w)

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1.5, X[:, 1].max() + 1.5
    x1_line = np.linspace(x1_min, x1_max, 100)

    # Decision boundary w.x+b=0 and the two margin lines w.x+b=+-1,
    # each solved for x2 as a function of x1 (w2 != 0 for this data).
    def boundary(offset):
        return -(w[0] * x1_line + b - offset) / w[1]

    ax.scatter(X[y == -1, 0], X[y == -1, 1], color="tab:blue", label="y=-1")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], color="tab:orange", label="y=+1")
    ax.scatter(X[is_sv, 0], X[is_sv, 1], s=180, facecolors="none",
               edgecolors="black", linewidths=1.5, label="support vectors")

    ax.plot(x1_line, boundary(0), color="black", linewidth=2, label="$w^Tx+b=0$")
    ax.plot(x1_line, boundary(1), color="grey", linestyle="--", linewidth=1)
    ax.plot(x1_line, boundary(-1), color="grey", linestyle="--", linewidth=1)

    # Normal vector w, drawn from the point on the boundary closest to
    # the origin, i.e. x0 = -b*w/||w||^2 (the boundary's foot of
    # perpendicular from the origin). Length is fixed relative to the
    # plot's own data range (not the margin itself), so the arrow stays
    # legible and within the panel regardless of how wide the margin is.
    x0_point = -b * w / np.dot(w, w)
    w_unit = w / np.linalg.norm(w)
    arrow_len = 0.18 * (x1_max - x1_min)
    arrow_end = x0_point + arrow_len * w_unit
    ax.annotate("", xy=arrow_end, xytext=x0_point,
                arrowprops=dict(arrowstyle="->", color="red", linewidth=2))
    ax.text(*(x0_point + arrow_len * w_unit * 1.25), "$w$", color="red", fontsize=12)

    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(f"generation gap={gap}  |  fitted margin={margin:.3f}  |  "
                 f"$\\|w\\|$={np.linalg.norm(w):.3f}")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_aspect("equal")


def plot_qp_geometry(ax, X, y, w_opt, b_opt):
    """Contour plot of the QP objective (1/2)||w||^2 directly in
    (w1, w2)-space, with b fixed at its optimal value b_opt, together
    with the linear constraints from the active support vectors. The
    optimal w sits where the smallest feasible objective contour
    touches the constraint boundary."""
    is_sv = find_support_vectors(X, y, w_opt, b_opt)

    # Center the view so it always includes the origin (contours of
    # (1/2)||w||^2 are circles centred there) as well as w*, with
    # enough margin to see the circles' curvature clearly - otherwise,
    # whenever w* is far from the origin, the view can clip to a
    # sliver of the circles that looks almost like straight lines.
    pad = 0.35 * max(abs(w_opt[0]), abs(w_opt[1]))
    w1_range = np.linspace(min(0, w_opt[0]) - pad, max(0, w_opt[0]) + pad, 300)
    w2_range = np.linspace(min(0, w_opt[1]) - pad, max(0, w_opt[1]) + pad, 300)
    W1, W2 = np.meshgrid(w1_range, w2_range)
    objective_grid = 0.5 * (W1 ** 2 + W2 ** 2)

    contours = ax.contour(W1, W2, objective_grid, levels=10, cmap="Greys", linewidths=1)
    ax.clabel(contours, inline=True, fontsize=7, fmt="%.2f")

    # Feasible region: for every point (support vector or not), the
    # constraint y_i*(w.x_i+b_opt) >= 1 must hold; each is a half-plane
    # in (w1,w2) for fixed b_opt. Shade the infeasible side of every
    # constraint lightly, and draw the ACTIVE (support-vector) ones
    # as solid lines - these are the constraints touching the
    # optimum.
    for i in range(len(y)):
        # y_i*(w1*x_i1 + w2*x_i2 + b_opt) = 1  =>  w2 = (1/y_i - b_opt - w1*x_i1*y_i) / (x_i2*y_i)
        # solve directly for the boundary line w2(w1):
        xi, yi = X[i], y[i]
        if abs(xi[1]) < 1e-8:
            continue
        w2_line = (1 - yi * b_opt - yi * xi[0] * w1_range) / (yi * xi[1])
        style = dict(color="tab:red", linewidth=2) if is_sv[i] else dict(color="tab:blue", linewidth=0.6, alpha=0.4)
        ax.plot(w1_range, w2_line, **style)

    ax.scatter(*w_opt, color="black", s=80, zorder=5, label="optimal $w^*$")
    ax.set_xlim(w1_range.min(), w1_range.max())
    ax.set_ylim(w2_range.min(), w2_range.max())
    ax.set_xlabel("$w_1$")
    ax.set_ylabel("$w_2$")
    ax.set_title("QP objective contours (grey) and constraints\n"
                 "(red = active/support-vector constraints, at $b=b^*$)")
    ax.legend(fontsize=9, loc="upper left")


def generate_data_with_noise(n_per_class, gap, noise_frac, spread=SPREAD, seed=SEED):
    """Same generator as generate_separable_data, plus a NOISE_FRAC
    fraction of points with their label flipped after generation. This
    is what breaks linear separability: some flipped points end up on
    the wrong side of where a clean boundary would sit, so no
    hyperplane can satisfy every hard-margin constraint any more."""
    X, y = generate_separable_data(n_per_class, gap, spread, seed)
    rng = np.random.default_rng(seed + 1000)  # separate stream from the data itself
    n = len(y)
    n_flip = int(round(noise_frac * n))
    flip_idx = rng.choice(n, size=n_flip, replace=False)
    y = y.copy()
    y[flip_idx] *= -1
    return X, y


def fit_svm_soft(X, y, C, x0=None):
    """Solve the soft-margin SVM primal QP directly:
        min_{w,b,xi} 0.5*||w||^2 + C*sum(xi)
        s.t.  y_i*(w.x_i+b) - 1 + xi_i >= 0   for all i
              xi_i >= 0                        for all i
    via SLSQP. Returns (w, b, xi).

    C controls the trade-off between margin width and how much slack
    (constraint violation) is tolerated: large C penalises slack
    heavily (behaves close to hard-margin, low bias/high variance);
    small C tolerates more slack in exchange for a wider margin
    (higher bias/lower variance) - the same bias-variance trade-off
    explored for regularisation earlier in this course, with C
    playing (inversely) the role lambda played there.

    x0, if given, warm-starts the solver from a previous solution
    (e.g. the fit at a neighbouring C) - SLSQP is a local optimiser,
    and cold-starting from all-zeros can occasionally fail to converge
    for large C, where the optimal xi are all near zero and the
    objective's slack term dominates sharply."""
    n, d = X.shape
    n_params = d + 1 + n  # w, b, xi

    def objective(params):
        w, xi = params[:d], params[d + 1:]
        return 0.5 * np.dot(w, w) + C * np.sum(xi)

    def objective_grad(params):
        w = params[:d]
        grad_xi = np.full(n, C)
        return np.concatenate([w, [0.0], grad_xi])

    def constraint_fn(params):
        w, b, xi = params[:d], params[d], params[d + 1:]
        return y * (X @ w + b) - 1.0 + xi

    def constraint_jac(params):
        J_w = y[:, None] * X
        J_b = y[:, None]
        J_xi = np.eye(n)
        return np.hstack([J_w, J_b, J_xi])

    bounds = [(None, None)] * (d + 1) + [(0, None)] * n  # xi_i >= 0

    starts = [x0] if x0 is not None else []
    starts.append(np.zeros(n_params))
    # A slack-heavy start (xi_i = max(0, 1 - y_i*(w.x_i+b)) at w=b=0,
    # i.e. xi_i=1) is also tried as a fallback, since it is always
    # feasible and sidesteps the bad-linesearch failure mode entirely.
    fallback = np.zeros(n_params)
    fallback[d + 1:] = 1.0
    starts.append(fallback)

    last_error = None
    for start in starts:
        result = minimize(
            objective, start, jac=objective_grad, method="SLSQP", bounds=bounds,
            constraints=[{"type": "ineq", "fun": constraint_fn, "jac": constraint_jac}],
            options={"maxiter": 2000, "ftol": 1e-10},
        )
        if result.success:
            w, b, xi = result.x[:d], result.x[d], result.x[d + 1:]
            xi = np.clip(xi, 0, None)  # guard against tiny negative numerical noise
            return w, b, xi
        last_error = result.message

    # SLSQP failed from every start. This tends to happen on larger,
    # harder problems (more points, more slack) at extreme C - SLSQP is
    # a general local optimiser, not a dedicated QP solver, and its line
    # search can fail on larger, more ill-conditioned problems even
    # though a solution exists. trust-constr is slower per iteration
    # but considerably more robust for larger bound-and-inequality-constrained 
    # QP, so it is tried as a fallback before giving up.
    
    from scipy.optimize import NonlinearConstraint, Bounds as ScipyBounds
    scipy_bounds = ScipyBounds([-np.inf] * (d + 1) + [0] * n, [np.inf] * n_params)
    nlc = NonlinearConstraint(constraint_fn, 0, np.inf, jac=constraint_jac)
    result = minimize(
        objective, starts[0], jac=objective_grad, method="trust-constr",
        bounds=scipy_bounds, constraints=[nlc],
        options={"maxiter": 3000, "gtol": 1e-8, "xtol": 1e-10},
    )
    if result.success:
        w, b, xi = result.x[:d], result.x[d], result.x[d + 1:]
        xi = np.clip(xi, 0, None)
        return w, b, xi

    raise RuntimeError(
        f"Soft-margin SVM QP did not converge for C={C} with either SLSQP "
        f"(message: {last_error}) or trust-constr (message: {result.message}). "
        f"Try a smaller dataset, or a less extreme C."
    )


def classify_points_soft(xi, tol=1e-4):
    """Categorise each point by its slack value:
        on-margin support vector : xi ~= 0, sits on a margin line
        margin violation         : 0 < xi <= 1, correct side, inside the margin
        misclassified            : xi > 1, on the wrong side of the boundary
    (points with xi essentially 0 AND not on the margin are just
    ordinary, correctly-classified points outside the margin)."""
    categories = np.full(len(xi), "ordinary", dtype=object)
    categories[(xi > tol) & (xi <= 1 + tol)] = "margin violation"
    categories[xi > 1 + tol] = "misclassified"
    return categories


def accuracy(X, y, w, b):
    pred = np.sign(X @ w + b)
    pred[pred == 0] = 1
    return np.mean(pred == y)


def split_train_holdout_test(X, y, frac_train=0.6, frac_holdout=0.2, seed=SEED):
    """Random split into train/holdout/test. Holdout is used only for
    selecting C; test is held out entirely until the final evaluation.
    """
    n = len(y)
    rng = np.random.default_rng(seed + 2000)
    idx = rng.permutation(n)
    n_train = int(round(frac_train * n))
    n_holdout = int(round(frac_holdout * n))
    train_idx = idx[:n_train]
    holdout_idx = idx[n_train:n_train + n_holdout]
    test_idx = idx[n_train + n_holdout:]
    return (
        (X[train_idx], y[train_idx]),
        (X[holdout_idx], y[holdout_idx]),
        (X[test_idx], y[test_idx]),
    )


def plot_soft_svm(ax, X, y, w, b, xi, title):
    categories = classify_points_soft(xi)

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1.5, X[:, 1].max() + 1.5
    x1_line = np.linspace(x1_min, x1_max, 100)

    def boundary(offset):
        return -(w[0] * x1_line + b - offset) / w[1]

    markers = {"ordinary": "o", "margin violation": "s", "misclassified": "X"}
    sizes = {"ordinary": 30, "margin violation": 70, "misclassified": 90}
    for cat in ["ordinary", "margin violation", "misclassified"]:
        mask = categories == cat
        for cls, color in [(-1, "tab:blue"), (1, "tab:orange")]:
            sel = mask & (y == cls)
            if sel.sum() == 0:
                continue
            ax.scatter(X[sel, 0], X[sel, 1], color=color, marker=markers[cat],
                       s=sizes[cat], edgecolors="black" if cat != "ordinary" else "none",
                       linewidths=1, label=f"y={cls:+d}, {cat}" if cls == -1 else None)

    ax.plot(x1_line, boundary(0), color="black", linewidth=2, label="$w^Tx+b=0$")
    ax.plot(x1_line, boundary(1), color="grey", linestyle="--", linewidth=1)
    ax.plot(x1_line, boundary(-1), color="grey", linestyle="--", linewidth=1)

    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.legend(fontsize=7, loc="lower right", markerscale=0.8)
    ax.set_aspect("equal")


def run_soft_margin_demo():
    X, y = generate_data_with_noise(N_PER_CLASS_SOFT, GAP_SOFT, NOISE_FRAC)
    (X_train, y_train), (X_hold, y_hold), (X_test, y_test) = split_train_holdout_test(X, y)
    print(f"\nSoft-margin SVM: {len(y)} points total "
          f"(train={len(y_train)}, holdout={len(y_hold)}, test={len(y_test)}), "
          f"noise_frac={NOISE_FRAC}")

    holdout_accs = []
    fits = []
    prev_x0 = None
    for C in C_GRID:
        w, b, xi = fit_svm_soft(X_train, y_train, C, x0=prev_x0)
        prev_x0 = np.concatenate([w, [b], xi])
        acc_train = accuracy(X_train, y_train, w, b)
        acc_hold = accuracy(X_hold, y_hold, w, b)
        holdout_accs.append(acc_hold)
        fits.append((w, b, xi))
        n_viol = (xi > 1e-4).sum()
        print(f"  C={C:>8}: train acc={acc_train:.3f}  holdout acc={acc_hold:.3f}  "
              f"points with slack>0: {n_viol}/{len(y_train)}")

    best_idx = int(np.argmax(holdout_accs))
    best_C = C_GRID[best_idx]
    print(f"Selected C={best_C} (highest holdout accuracy = {holdout_accs[best_idx]:.3f})")

    # Refit on train+holdout with the selected C, evaluate on the
    # untouched test set - the final, honest performance estimate.
    X_trainval = np.vstack([X_train, X_hold])
    y_trainval = np.concatenate([y_train, y_hold])
    w_final, b_final, xi_final = fit_svm_soft(X_trainval, y_trainval, best_C)
    acc_test = accuracy(X_test, y_test, w_final, b_final)
    print(f"Final model (C={best_C}) test accuracy: {acc_test:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].plot(range(len(C_GRID)), holdout_accs, marker="o", color="black")
    axes[0].set_xticks(range(len(C_GRID)))
    axes[0].set_xticklabels([str(c) for c in C_GRID])
    axes[0].axvline(best_idx, color="red", linestyle="--", linewidth=1,
                     label=f"selected C={best_C}")
    axes[0].set_xlabel("C")
    axes[0].set_ylabel("holdout accuracy")
    axes[0].set_title("Model selection: holdout accuracy vs. C")
    axes[0].legend(fontsize=9)

    plot_soft_svm(axes[1], X_trainval, y_trainval, w_final, b_final, xi_final,
                  f"Soft-margin SVM (C={best_C}), train+holdout\ntest accuracy={acc_test:.3f}")

    fig.suptitle(f"Soft-margin SVM with slack variables (noise_frac={NOISE_FRAC})")
    fig.tight_layout()
    fig.savefig("svm_soft_margin.png", dpi=150)
    print("Saved plot to svm_soft_margin.png")


def main():
    fig1, axes = plt.subplots(1, len(GAPS), figsize=(6 * len(GAPS), 5.5))
    fitted = []
    for ax, gap in zip(axes, GAPS):
        X, y = generate_separable_data(N_PER_CLASS, gap)
        w, b = fit_svm_primal(X, y)
        margin = 2.0 / np.linalg.norm(w)
        print(f"gap={gap:>4}: w={np.round(w, 4)}  b={b:.4f}  "
              f"margin={margin:.4f}  #support vectors={find_support_vectors(X, y, w, b).sum()}")
        plot_svm(ax, X, y, w, b, gap)
        fitted.append((X, y, w, b))

    fig1.suptitle("Linear (hard-margin) SVM: effect of class separation on the fitted margin")
    fig1.tight_layout()
    fig1.savefig("svm_margins.png", dpi=150)
    print("Saved plot to svm_margins.png")

    # Second figure: QP geometry, for the middle (representative) gap.
    X_mid, y_mid, w_mid, b_mid = fitted[len(GAPS) // 2]
    fig2, ax2 = plt.subplots(figsize=(7, 6.5))
    plot_qp_geometry(ax2, X_mid, y_mid, w_mid, b_mid)
    fig2.suptitle(f"How the optimum is found (gap={GAPS[len(GAPS)//2]})")
    fig2.tight_layout()
    fig2.savefig("svm_qp_geometry.png", dpi=150)
    print("Saved plot to svm_qp_geometry.png")

    run_soft_margin_demo()


if __name__ == "__main__":
    main()

### Visualisation for 3-dimensions

In [ ]:
"""
Demo of a linear hard-margin SVM on three features (3D).

Model:      y_hat = sign(w^T x + b)
Objective:  min_{w,b}  (1/2) ||w||^2
Constraints: y_i (w^T x_i + b) >= 1   for all i

Solved directly as a constrained quadratic program via
scipy.optimize.minimize (SLSQP). The solver code is dimension-agnostic 
(it only uses X's number of columns, d); only the data generation and 
plotting arespecific to 3 features here, drawing the decision boundary and 
margins as PLANES in (x1, x2, x3)-space.

ELEV and AZIM control the 3D camera angle (matplotlib's ax.view_init),
so the view can be rotated.
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3D projection)
from scipy.optimize import minimize

N_PER_CLASS = 12
GAP = 3.0       # centre-to-centre distance between the two classes
SPREAD = 0.5    # within-class scatter
SEED = 0

ELEV = 15       # camera elevation angle (degrees above the x1-x2 plane)
AZIM = 75      # camera azimuth angle (degrees, rotation around the z-axis)


def generate_data(n_per_class=N_PER_CLASS, gap=GAP, spread=SPREAD, seed=SEED):
    rng = np.random.default_rng(seed)
    X_neg = rng.normal([-gap / 2, 0.0, 0.0], spread, size=(n_per_class, 3))
    X_pos = rng.normal([gap / 2, 0.0, 0.0], spread, size=(n_per_class, 3))
    X = np.vstack([X_neg, X_pos])
    y = np.concatenate([-np.ones(n_per_class), np.ones(n_per_class)])
    return X, y


def fit_hard_margin_svm(X, y):
    n, d = X.shape

    def objective(params):
        w = params[:d]
        return 0.5 * np.dot(w, w)

    def objective_grad(params):
        w = params[:d]
        return np.concatenate([w, [0.0]])

    def constraint_fn(params):
        w, b = params[:d], params[d]
        return y * (X @ w + b) - 1.0

    def constraint_jac(params):
        return np.column_stack([y[:, None] * X, y])

    result = minimize(
        objective, np.zeros(d + 1), jac=objective_grad, method="SLSQP",
        constraints=[{"type": "ineq", "fun": constraint_fn, "jac": constraint_jac}],
        options={"maxiter": 500, "ftol": 1e-12},
    )
    if not result.success:
        raise RuntimeError(f"SVM QP did not converge: {result.message}")
    return result.x[:d], result.x[d]


def plane(x1_grid, x2_grid, w, b, offset):
    """Solve w1*x1 + w2*x2 + w3*x3 + b = offset for x3, over a grid of
    (x1, x2) - requires w3 != 0, which holds generically here."""
    return (offset - b - w[0] * x1_grid - w[1] * x2_grid) / w[2]


def main():
    X, y = generate_data()
    w, b = fit_hard_margin_svm(X, y)
    margin = 2.0 / np.linalg.norm(w)

    is_sv = np.abs(y * (X @ w + b) - 1.0) < 1e-3
    print(f"w = {np.round(w, 4)}")
    print(f"b = {b:.4f}")
    print(f"margin = {margin:.4f}")
    print(f"support vectors: {is_sv.sum()}")

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    x3_min, x3_max = X[:, 2].min() - 1, X[:, 2].max() + 1
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(x1_min, x1_max, 15), np.linspace(x2_min, x2_max, 15)
    )

    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(projection="3d")

    ax.scatter(X[y == -1, 0], X[y == -1, 1], X[y == -1, 2],
               color="tab:blue", label="y=-1", s=40)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], X[y == 1, 2],
               color="tab:orange", label="y=+1", s=40)
    ax.scatter(X[is_sv, 0], X[is_sv, 1], X[is_sv, 2],
               s=220, facecolors="none", edgecolors="black", linewidths=1.5,
               label="support vectors")

    ax.plot_surface(x1_grid, x2_grid, plane(x1_grid, x2_grid, w, b, 0),
                     color="black", alpha=0.35, linewidth=0)
    ax.plot_surface(x1_grid, x2_grid, plane(x1_grid, x2_grid, w, b, 1),
                     color="grey", alpha=0.15, linewidth=0)
    ax.plot_surface(x1_grid, x2_grid, plane(x1_grid, x2_grid, w, b, -1),
                     color="grey", alpha=0.15, linewidth=0)

    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)
    ax.set_zlim(x3_min, x3_max)
    ax.view_init(elev=ELEV, azim=AZIM)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_zlabel("$x_3$")
    ax.set_title(f"Hard-margin SVM, 3 features  "
                 f"(margin={margin:.3f}, $\\|w\\|$={np.linalg.norm(w):.3f})")
    ax.legend(loc="upper left", fontsize=9)
    fig.tight_layout()
    fig.savefig("svm_hard_margin_3d_demo.png", dpi=150)
    print("Saved plot to svm_hard_margin_3d_demo.png")


if __name__ == "__main__":
    main()